In [1]:
import pandas as pd

train = pd.read_csv('data/synthea_processed/train.csv')
val = pd.read_csv('data/synthea_processed/val.csv')
test = pd.read_csv('data/synthea_processed/test.csv')

print(train.shape, val.shape, test.shape)
print(train['label'].value_counts(normalize=True))
train.head()


(6405, 8) (800, 8) (802, 8)
label
0    0.95082
1    0.04918
Name: proportion, dtype: float64


,patient_id,age,gender,medication,conditions,allergies,label,reason
0,f4bfe7d7-53a4-4dd6-b404-c6af82a52f0b,12.334018,M,pseudoephedrine,acute viral pharyngitis (disorder)|concussion ...,NaN,0,NaN
1,1d74896f-fda4-4b30-892e-c486e5e32ca4,52.783025,M,acetaminophen,acute bacterial sinusitis (disorder)|acute bro...,NaN,0,NaN
2,8dd20f15-2beb-4568-83d4-2109123f5ffd,26.036961,F,acetaminophen,acute viral pharyngitis (disorder)|body mass i...,NaN,0,NaN
3,f1728f28-ef0c-4342-9216-a4a4e8375961,32.966461,F,acetaminophen,acute bacterial sinusitis (disorder)|acute bro...,NaN,0,NaN
4,846b0d5b-e5f5-4fd6-8ae4-7aa8e13a17c3,55.394935,M,valpro,acute bronchitis (disorder)|acute viral pharyn...,NaN,0,NaN


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Prepare columns
X_train = train.copy()
y_train = X_train.pop("label")

X_val = val.copy()
y_val = X_val.pop("label")

X_test = test.copy()
y_test = X_test.pop("label")

# Combine text fields into one
for df in (X_train, X_val, X_test):
    df["text"] = (
        df["medication"].fillna("") + " || "
        + df["conditions"].fillna("") + " || "
        + df["allergies"].fillna("")
    )

# Preprocess: text -> TFIDF, gender -> one-hot, age -> numeric
preprocess = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(min_df=2, ngram_range=(1, 2)), "text"),
        ("gender", OneHotEncoder(handle_unknown="ignore"), ["gender"]),
        ("age", Pipeline(steps=[("imp", SimpleImputer(strategy="median"))]), ["age"]),
    ],
    remainder="drop",
)

model = LogisticRegression(max_iter=2000, class_weight="balanced")

clf = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
clf.fit(X_train, y_train)

pred_val = clf.predict(X_val)
print("VAL RESULTS")
print(classification_report(y_val, pred_val, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_val, pred_val))

VAL RESULTS
              precision    recall  f1-score   support

           0     0.9971    0.8922    0.9417       761
           1     0.3109    0.9487    0.4684        39

    accuracy                         0.8950       800
   macro avg     0.6540    0.9205    0.7051       800
weighted avg     0.9636    0.8950    0.9187       800

Confusion matrix:
 [[679  82]
 [  2  37]]


In [3]:
pred_test = clf.predict(X_test)
print("TEST RESULTS")
print(classification_report(y_test, pred_test, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_test))

TEST RESULTS
              precision    recall  f1-score   support

           0     0.9956    0.9003    0.9456       762
           1     0.3274    0.9250    0.4837        40

    accuracy                         0.9015       802
   macro avg     0.6615    0.9126    0.7146       802
weighted avg     0.9623    0.9015    0.9225       802

Confusion matrix:
 [[686  76]
 [  3  37]]
